# Módulo 01 — Parser de EPUB

---

**Projeto**: Pergunte ao Livro  
**Arquivo de origem**: `src/parser.py`  
**Objetivo**: Entender como o sistema lê e extrai texto de arquivos `.epub`

---

## O que você vai aprender

1. O que é um arquivo EPUB e como ele é estruturado
2. Como usar a biblioteca `ebooklib` para ler EPUBs
3. Como extrair texto de HTML com `BeautifulSoup`
4. Como usar `re` (regex) para validar e extrair ISBNs
5. Como gerar um identificador único (`book_id`) para cada livro
6. Como funciona a função `parse_epub` que alimenta todo o pipeline

---

## 1. O que é um EPUB?

Um arquivo `.epub` é, na prática, um arquivo **ZIP** contendo:

- Vários arquivos **HTML/XHTML** (um por capítulo ou seção)
- Arquivos de metadados em **XML** (título, autor, ISBN, etc.)
- Imagens, CSS e outros recursos

```
meu_livro.epub
├── META-INF/
│   └── container.xml       ← diz onde está o arquivo principal
├── OEBPS/
│   ├── content.opf         ← metadados (título, autor, ISBN)
│   ├── chapter1.xhtml      ← texto do capítulo 1
│   ├── chapter2.xhtml      ← texto do capítulo 2
│   └── ...
└── mimetype
```

A biblioteca `ebooklib` abstrai todo esse ZIP e dá acesso fácil ao conteúdo.

## 2. Instalando as dependências

In [ ]:
# Descomente e execute se ainda não tiver instalado
# !pip install ebooklib beautifulsoup4 lxml

In [1]:
import re
import ebooklib
from ebooklib import epub
from bs4 import BeautifulSoup

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


## 3. Lendo um EPUB com ebooklib

A função principal da `ebooklib` é `epub.read_epub()`. Ela retorna um objeto `EpubBook` com métodos para acessar metadados e conteúdo.

In [2]:
# Caminho para o epub de exemplo incluído no projeto
EPUB_PATH = "../data/books/O Reverso Da Medalha.epub"

book = epub.read_epub(EPUB_PATH)
print(type(book))

<class 'ebooklib.epub.EpubBook'>


### 3.1 Metadados Dublin Core (DC)

O padrão EPUB usa o schema **Dublin Core** para metadados. Os campos mais importantes são:

| Campo DC      | Significado          |
|---------------|----------------------|
| `title`       | Título do livro      |
| `creator`     | Autor(es)            |
| `identifier`  | ISBN ou outro ID     |
| `language`    | Idioma               |
| `publisher`   | Editora              |

`get_metadata("DC", campo)` retorna uma **lista de tuplas** `(valor, atributos)`.

In [3]:
# Cada chamada retorna lista de tuplas (valor, atributos_extras)
titles = book.get_metadata("DC", "title")
creators = book.get_metadata("DC", "creator")
identifiers = book.get_metadata("DC", "identifier")

print("titles:", titles)
print()
print("creators:", creators)
print()
print("identifiers:", identifiers)

titles: [('O Reverso Da Medalha - vol.1', {})]

creators: [('Sidney Sheldon', {'{http://www.idpf.org/2007/opf}file-as': 'Sheldon, Sidney', '{http://www.idpf.org/2007/opf}role': 'aut'})]

identifiers: [('ed25eafa-5f86-4ec7-847f-969e50bad059', {'id': 'uuid_id', '{http://www.idpf.org/2007/opf}scheme': 'uuid'}), ('ed25eafa-5f86-4ec7-847f-969e50bad059', {'{http://www.idpf.org/2007/opf}scheme': 'calibre'})]


In [4]:
# Para pegar só o valor (primeiro elemento da primeira tupla):
titulo = titles[0][0] if titles else None
autor  = creators[0][0] if creators else None

print(f"Título : {titulo}")
print(f"Autor  : {autor}")

Título : O Reverso Da Medalha - vol.1
Autor  : Sidney Sheldon


## 4. Extraindo o ISBN com Regex

O ISBN-13 é um número de 13 dígitos que começa com **978** ou **979**.  
O padrão regex `97[89]\d{10}` significa:

- `97`     — começa com "97"
- `[89]`   — seguido de "8" ou "9"
- `\d{10}` — seguido de exatamente 10 dígitos

Antes de testar o regex, precisamos limpar o valor: remover hífens e espaços.

In [5]:
# Exemplos de valores que podem vir no campo 'identifier'
exemplos = [
    "978-85-1234-567-0",   # ISBN com hífens — válido após limpar
    "urn:isbn:9788512345670",  # ISBN com prefixo
    "ABCDE12345",          # ID qualquer — não é ISBN
]

for ex in exemplos:
    limpo = ex.replace("-", "").replace(" ", "")
    match = re.search(r"97[89]\d{10}", limpo)
    print(f"{ex!r:40} → limpo: {limpo!r:20} → match: {match.group() if match else 'None'}")

'978-85-1234-567-0'                      → limpo: '9788512345670'      → match: 9788512345670
'urn:isbn:9788512345670'                 → limpo: 'urn:isbn:9788512345670' → match: 9788512345670
'ABCDE12345'                             → limpo: 'ABCDE12345'         → match: None


In [6]:
identifiers

[('ed25eafa-5f86-4ec7-847f-969e50bad059',
  {'id': 'uuid_id', '{http://www.idpf.org/2007/opf}scheme': 'uuid'}),
 ('ed25eafa-5f86-4ec7-847f-969e50bad059',
  {'{http://www.idpf.org/2007/opf}scheme': 'calibre'})]

In [7]:
# Aplicando ao livro real — identifiers é lista de tuplas (valor, atributos)
isbn = next(
    (str(v) for v, _ in identifiers
     if re.match(r"97[89]\d{10}", str(v).replace("-", "").replace(" ", ""))),
    None,  # valor padrão caso não encontre nenhum ISBN
)

print(f"ISBN encontrado: {isbn}")

ISBN encontrado: None


> **`next(iterador, default)`** — percorre o iterador e retorna o **primeiro** elemento que satisfaz a condição, ou `default` se nenhum satisfizer. Muito mais eficiente que criar uma lista inteira com list comprehension quando só precisamos do primeiro resultado.

## 5. Gerando o `book_id`

O `book_id` é um identificador único e legível para cada livro no sistema.  
A lógica em `_generate_book_id` tem dois caminhos:

1. **Se tiver ISBN** → normaliza o ISBN como slug (ex: `978-85-xxx` → `978-85-xxx`)
2. **Fallback** → combina título + autor e transforma em slug

A transformação de slug usa `re.sub(r"[^a-z0-9]+", "-", texto)` para substituir qualquer coisa que **não seja** letra minúscula ou número por um hífen.

In [8]:
# Entendendo a regex de slug
def para_slug(texto: str, max_len: int = 64) -> str:
    """Transforma qualquer texto em um slug seguro para usar como ID."""
    return re.sub(r"[^a-z0-9]+", "-", texto.lower()).strip("-")[:max_len]

exemplos = [
    "O Reverso da Medalha",
    "978-85-200-1234-5",
    "Título Com Acentuação & Símbolos!",
]

for ex in exemplos:
    print(f"{ex!r:40} → {para_slug(ex)!r}")

'O Reverso da Medalha'                   → 'o-reverso-da-medalha'
'978-85-200-1234-5'                      → '978-85-200-1234-5'
'Título Com Acentuação & Símbolos!'      → 't-tulo-com-acentua-o-s-mbolos'


In [9]:
# A função completa do projeto (copiada de src/parser.py)
def _generate_book_id(book) -> str:
    # Tenta usar o ISBN primeiro
    identifiers = book.get_metadata("DC", "identifier") or []
    for value, _ in identifiers:
        val = str(value).replace("-", "").replace(" ", "")
        if re.match(r"97[89]\d{10}", val):
            return re.sub(r"[^a-z0-9]+", "-", str(value).lower()).strip("-")[:64]

    # Fallback: título + autor
    titles   = book.get_metadata("DC", "title")   or []
    creators = book.get_metadata("DC", "creator") or []
    title  = titles[0][0]   if titles   else "unknown"
    author = creators[0][0] if creators else "unknown"
    raw = f"{title}-{author}".lower()
    return re.sub(r"[^a-z0-9]+", "-", raw).strip("-")[:64]


book_id = _generate_book_id(book)
print(f"book_id gerado: {book_id!r}")
print(f"Tamanho       : {len(book_id)} chars")

book_id gerado: 'o-reverso-da-medalha-vol-1-sidney-sheldon'
Tamanho       : 41 chars


## 6. Extraindo o texto dos capítulos com BeautifulSoup

Cada capítulo do EPUB é um documento HTML/XHTML. O `ebooklib` dá acesso a esses documentos como objetos `EpubItem`.  

Para verificar se um item é um documento de texto (e não uma imagem ou CSS), usamos `item.get_type() == ebooklib.ITEM_DOCUMENT`.

In [10]:
# Listando todos os tipos de itens presentes no EPUB
from collections import Counter

tipos = Counter(item.get_type() for item in book.get_items())
print("Tipos de itens no EPUB:")
for tipo, qtd in tipos.items():
    # Traduzindo os códigos numéricos para nomes legíveis
    nome = {v: k for k, v in vars(ebooklib).items() if k.startswith("ITEM_")}.get(tipo, tipo)
    print(f"  {nome:30} → {qtd} item(s)")

Tipos de itens no EPUB:
  ITEM_IMAGE                     → 1 item(s)
  ITEM_DOCUMENT                  → 10 item(s)
  ITEM_STYLE                     → 2 item(s)
  ITEM_NAVIGATION                → 1 item(s)


In [11]:
# Pegando apenas os documentos de texto
documentos = [item for item in book.get_items() if item.get_type() == ebooklib.ITEM_DOCUMENT]
print(f"Total de documentos: {len(documentos)}")

# Inspecionando o primeiro documento
primeiro = documentos[0]
print(f"\nID       : {primeiro.get_id()}")
print(f"\nConteúdo (primeiros 500 chars do HTML):")
print(primeiro.get_content()[:500].decode("utf-8"))

Total de documentos: 10

ID       : id19

Conteúdo (primeiros 500 chars do HTML):
<?xml version='1.0' encoding='utf-8'?>
<!DOCTYPE html>
<html xmlns="http://www.w3.org/1999/xhtml" xmlns:epub="http://www.idpf.org/2007/ops" epub:prefix="z3998: http://www.daisy.org/z3998/2012/vocab/structure/#" lang="en" xml:lang="en">
  <head/>
  <body><div class="P-P">
<div class="G-fr"><img src="Pictures/100000000000013C000001EA5110A152.jpg" alt="" class="calibre1"/></div>

<span class="S-T"/> </div>
</body>
</html>



### 6.1 Extraindo texto limpo com BeautifulSoup

`BeautifulSoup` parseia o HTML e `get_text()` retorna apenas o texto puro, sem tags.

In [12]:
# Parseando o HTML do primeiro documento
soup = BeautifulSoup(primeiro.get_content(), "html.parser")
texto = soup.get_text().strip()

print(f"Tamanho do texto: {len(texto)} chars")
print(f"\nPrimeiros 300 chars:")
print(texto[:300])

Tamanho do texto: 0 chars

Primeiros 300 chars:



In [13]:
# Por que filtramos documentos com menos de 100 chars?
# Alguns "documentos" são só páginas em branco, capas ou folhas de rosto.

tamanhos = [(item.get_id(), len(BeautifulSoup(item.get_content(), "html.parser").get_text().strip()))
            for item in documentos]

print("Tamanhos dos documentos:")
for doc_id, tamanho in tamanhos:
    marcador = "✗ IGNORADO" if tamanho < 100 else "✓"
    print(f"  {marcador} {doc_id:35} → {tamanho:6} chars")

Tamanhos dos documentos:
  ✗ IGNORADO id19                                →      0 chars
  ✓ id18                                →   8904 chars
  ✓ id17                                → 133497 chars
  ✓ id16                                → 132759 chars
  ✓ id15                                →  76618 chars
  ✓ id14                                →  49991 chars
  ✓ id13                                → 102999 chars
  ✓ id12                                → 149399 chars
  ✓ id11                                → 130018 chars
  ✗ IGNORADO titlepage                           →      0 chars


## 7. A função completa `parse_epub`

Vamos agora ver a função que une tudo isso e entender seu retorno.

In [14]:
def parse_epub(file_path: str) -> tuple[dict, list[dict]]:
    """Lê um arquivo EPUB e retorna metadados do livro + lista de capítulos."""
    book = epub.read_epub(file_path)

    titles      = book.get_metadata("DC", "title")      or []
    creators    = book.get_metadata("DC", "creator")    or []
    identifiers = book.get_metadata("DC", "identifier") or []

    isbn = next(
        (str(v) for v, _ in identifiers
         if re.match(r"97[89]\d{10}", str(v).replace("-", "").replace(" ", ""))),
        None,
    )

    book_meta = {
        "id":     _generate_book_id(book),
        "title":  titles[0][0]   if titles   else None,
        "author": creators[0][0] if creators else None,
        "isbn":   isbn,
    }

    chapters = []
    for item in book.get_items():
        if item.get_type() == ebooklib.ITEM_DOCUMENT:
            soup = BeautifulSoup(item.get_content(), "html.parser")
            text = soup.get_text().strip()
            if len(text) < 100:
                continue
            chapters.append({"id": item.get_id(), "text": text})

    return book_meta, chapters

In [15]:
# Executando e inspecionando o retorno
book_meta, chapters = parse_epub(EPUB_PATH)

print("=== METADADOS ===")
for k, v in book_meta.items():
    print(f"  {k:8}: {v}")

print(f"\n=== CAPÍTULOS ===")
print(f"  Total de capítulos: {len(chapters)}")
print(f"  Campos de cada capítulo: {list(chapters[0].keys())}")

print(f"\n  Capítulo 1 — ID: {chapters[0]['id']}")
print(f"  Tamanho do texto: {len(chapters[0]['text'])} chars")
print(f"  Prévia: {chapters[0]['text'][:200]!r}")

=== METADADOS ===
  id      : o-reverso-da-medalha-vol-1-sidney-sheldon
  title   : O Reverso Da Medalha - vol.1
  author  : Sidney Sheldon
  isbn    : None

=== CAPÍTULOS ===
  Total de capítulos: 8
  Campos de cada capítulo: ['id', 'text']

  Capítulo 1 — ID: id18
  Tamanho do texto: 8904 chars
  Prévia: 'O Reverso Da Medalha SIDNEY SHELDON \n Título original; MASTER OF THE GAME Tradução de: EDUARDO SALÓ \n  A meu irmão Richard, Coração de Leão A minha gratidão vai para Miss Geraldine Hunter, pela sua pa'


## 8. Como o parser se encaixa no pipeline

```
arquivo.epub
    │
    ▼
parse_epub()          ← você está aqui
    │
    ├── book_meta: { id, title, author, isbn }
    │
    └── chapters: [ { id, text }, { id, text }, ... ]
                          │
                          ▼
                   chunk_chapters()    (Módulo 02)
                          │
                          ▼
                   embed_chunks()      (Módulo 03)
                          │
                          ▼
                   store em ChromaDB   (Módulo 04)
```

O `book_meta` é salvo separadamente (banco SQLite via `book_catalog.py`) e o `chapters` entra no pipeline de chunks → embeddings → retrieval.

---

## Exercícios

---

### **`E1`** Inspecionando os metadados

Usando o objeto `book` já carregado, extraia e exiba:
- O idioma do livro (campo DC `language`)
- O editor/publisher (campo DC `publisher`) — use `None` como padrão se não existir


In [16]:
# E1 — Seu código aqui
languages = book.get_metadata("DC", "language") or []
publishers = book.get_metadata("DC","publisher") or []

idioma    = languages[0][0]  if languages  else None
publisher = publishers[0][0] if publishers else None

print(f"Idioma   : {idioma}")
print(f"Publisher: {publisher}")

Idioma   : pt
Publisher: Top Livros


---

### **`E2`** Analisando os capítulos

Usando a lista `chapters` retornada por `parse_epub`:

a) Qual capítulo tem o **maior** número de caracteres? Exiba o `id` e o tamanho.  
b) Qual capítulo tem o **menor** número de caracteres (dentre os que passaram pelo filtro de 100 chars)? Exiba o `id` e o tamanho.  
c) Qual é a média de tamanho de texto por capítulo?

In [ ]:
# E2 — Seu código aqui


---

### **`E3`** Validação de ISBN

Crie uma função chamada `is_valid_isbn13(value: str) -> bool` que retorne `True` se o valor (após remover hífens e espaços) for um ISBN-13 válido.

Teste com os valores abaixo:

```python
testes = [
    ("978-85-359-0277-5", True),
    ("979-10-91999-01-3", True),
    ("0-306-40615-2",     False),  # ISBN-10, não ISBN-13
    ("12345",             False),
    ("978852009",         False),  # só 9 dígitos após 978
]
```

In [ ]:
# E3 — Seu código aqui

def is_valid_isbn13(value: str) -> bool:
    # Seu código aqui
    pass


testes = [
    ("978-85-359-0277-5", True),
    ("979-10-91999-01-3", True),
    ("0-306-40615-2",     False),
    ("12345",             False),
    ("978852009",         False),
]

for valor, esperado in testes:
    resultado = is_valid_isbn13(valor)
    status = "✓" if resultado == esperado else "✗ ERRO"
    print(f"{status} is_valid_isbn13({valor!r}) = {resultado} (esperado: {esperado})")

---

### **`E4`** Gerando slugs

Crie uma função `gerar_book_id(titulo: str, autor: str) -> str` que gere um `book_id` no formato `titulo-autor` (slug, máximo 64 chars), sem usar o objeto `book`.

Teste:
```python
gerar_book_id("O Senhor dos Anéis", "J.R.R. Tolkien")
# Esperado: 'o-senhor-dos-an-is-j-r-r-tolkien'

gerar_book_id("A" * 100, "Autor")  
# O resultado deve ter no máximo 64 chars
```

In [ ]:
# E4 — Seu código aqui

def gerar_book_id(titulo: str, autor: str) -> str:
    # Seu código aqui
    pass


print(gerar_book_id("O Senhor dos Anéis", "J.R.R. Tolkien"))
resultado_longo = gerar_book_id("A" * 100, "Autor")
print(resultado_longo)
print(f"Tamanho: {len(resultado_longo)} chars")

---

### **`E5`** Desafio — Estatísticas do livro

Crie uma função `book_stats(file_path: str) -> dict` que receba o caminho de um EPUB e retorne um dicionário com:

```python
{
    "title":          "...",
    "author":         "...",
    "n_chapters":     int,    # número de capítulos (após filtro de 100 chars)
    "total_chars":    int,    # total de caracteres no livro
    "avg_chars":      float,  # média de chars por capítulo
    "largest_chapter": str,   # id do capítulo com mais chars
}
```

Dica: use as funções `parse_epub` e `_generate_book_id` que você já conhece.

In [ ]:
# E5 — Seu código aqui

def book_stats(file_path: str) -> dict:
    # Seu código aqui
    pass


stats = book_stats(EPUB_PATH)
if stats:
    for k, v in stats.items():
        print(f"  {k:20}: {v}")

---

## Gabarito

> Execute as células abaixo apenas depois de tentar resolver os exercícios por conta própria.

In [ ]:
# Gabarito E1
languages  = book.get_metadata("DC", "language")  or []
publishers = book.get_metadata("DC", "publisher") or []

idioma    = languages[0][0]  if languages  else None
publisher = publishers[0][0] if publishers else None

print(f"Idioma   : {idioma}")
print(f"Publisher: {publisher}")

In [ ]:
# Gabarito E2
maior  = max(chapters, key=lambda c: len(c["text"]))
menor  = min(chapters, key=lambda c: len(c["text"]))
media  = sum(len(c["text"]) for c in chapters) / len(chapters)

print(f"Maior capítulo : {maior['id']} ({len(maior['text'])} chars)")
print(f"Menor capítulo : {menor['id']} ({len(menor['text'])} chars)")
print(f"Média por cap. : {media:.0f} chars")

In [ ]:
# Gabarito E3
def is_valid_isbn13(value: str) -> bool:
    limpo = value.replace("-", "").replace(" ", "")
    return bool(re.match(r"^97[89]\d{10}$", limpo))

for valor, esperado in testes:
    resultado = is_valid_isbn13(valor)
    status = "✓" if resultado == esperado else "✗ ERRO"
    print(f"{status} is_valid_isbn13({valor!r}) = {resultado}")

In [ ]:
# Gabarito E4
def gerar_book_id(titulo: str, autor: str) -> str:
    raw = f"{titulo}-{autor}".lower()
    return re.sub(r"[^a-z0-9]+", "-", raw).strip("-")[:64]

print(gerar_book_id("O Senhor dos Anéis", "J.R.R. Tolkien"))
resultado_longo = gerar_book_id("A" * 100, "Autor")
print(resultado_longo)
print(f"Tamanho: {len(resultado_longo)} chars")

In [ ]:
# Gabarito E5
def book_stats(file_path: str) -> dict:
    book_meta, chapters = parse_epub(file_path)
    tamanhos = [len(c["text"]) for c in chapters]
    maior_idx = tamanhos.index(max(tamanhos))
    return {
        "title":           book_meta["title"],
        "author":          book_meta["author"],
        "n_chapters":      len(chapters),
        "total_chars":     sum(tamanhos),
        "avg_chars":       sum(tamanhos) / len(tamanhos),
        "largest_chapter": chapters[maior_idx]["id"],
    }

stats = book_stats(EPUB_PATH)
for k, v in stats.items():
    print(f"  {k:20}: {v}")